In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(3,32),
            nn.Tanh(),
            nn.Linear(32,64),
            nn.Tanh(),
            nn.Linear(64,16),
            nn.Tanh(),
            nn.Linear(16,1)
        )
    
    def forward(self,t,y1,y2):
        x=torch.cat([t,y1,y2],dim=1)
        return self.net(x)
    def loss(self,t,y1,y2,r,K1,K2,sigma):
        t.requires_grad_(True)
        y1.requires_grad_(True)
        y2.requires_grad_(True)

        w1=self.forward(t,y1,y2)
        w2=self.forward(t,y2,y1)

        dw1=torch.autograd.grad(w1,y1,grad_outputs=torch.ones_like(w1),create_graph=True)[0]
        dw2=torch.autograd.grad(w1,y2,grad_outputs=torch.ones_like(w1),create_graph=True)[0]

        dw3=torch.autograd.grad(w1,t,grad_outputs=torch.ones_like(w1),create_graph=True)[0]

        d2wd11=torch.autograd.grad(dw1,y1,grad_outputs=torch.ones_like(w1),create_graph=True)[0]
        d2wd22=torch.autograd.grad(dw2,y2,grad_outputs=torch.ones_like(w1),create_graph=True)[0]

        val=(K1+K2)*dw3*(dw1-dw2)+r*(K1*y1+K2*y2)*(dw1**2-dw2**2)-r*dw1*(K1*w1+K2*w2)+r*dw2*(K2*w1+K1*w2)
        term=(K1*(y1**2)+K2*(y2**2))*(d2wd11*dw1-d2wd22*dw2)-(K1*(y2**2)+K2*(y1**2))*(d2wd11*dw2-d2wd22*dw1)
        val+=0.5*sigma*sigma*(term)

        return torch.mean(val**2)
    def basecase_loss(self,y1,y2,t0,c):
        t=torch.full_like(y1,t0)
        W1=self.forward(t,y1,y2)
        W2=self.forward(t,y2,y1)
        return torch.mean((W1-torch.relu(y1-c))**2+(W2-torch.relu(y2-c))**2)



In [3]:
def train(model):
    r=0.05
    K1=0.7
    K2=1-K1
    sigma=0.1
    C=100
    t0=1000
    epochs=5000
    batch_size=1000
    optimizer=optim.Adam(model.parameters(),lr=1e-3)

    for epoch in range(epochs):
        optimizer.zero_grad()
        t_colloc = torch.rand((batch_size, 1)) * t0
        y1_colloc = torch.rand((batch_size, 1)) * (2 * C) # Sample prices from 0 to 200
        y2_colloc = torch.rand((batch_size, 1)) * (2 * C)

        # Randomly sample points for the terminal boundary
        y1_term = torch.rand((batch_size, 1)) * (2 * C)
        y2_term = torch.rand((batch_size, 1)) * (2 * C)

        # Calculate losses
        loss_pde = model.loss(t_colloc, y1_colloc, y2_colloc, r, K1, K2, sigma)
        loss_term = model.basecase_loss(y1_term, y2_term, t0, C)

        # Total loss (you can weigh these differently if one dominates)
        total_loss = loss_pde + loss_term

        total_loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print(f"Epoch {epoch}: PDE Loss = {loss_pde.item():.6f}, Terminal Loss = {loss_term.item():.6f}")

    return model

In [4]:
pricing_function=PINN()

In [9]:
train(pricing_function)
torch.save(pricing_function.state_dict(), 'model.pth')

Epoch 0: PDE Loss = 2.538178, Terminal Loss = 0.370751
Epoch 500: PDE Loss = 2.454459, Terminal Loss = 0.678500
Epoch 1000: PDE Loss = 2.646894, Terminal Loss = 0.672392
Epoch 1500: PDE Loss = 2.445643, Terminal Loss = 1.148618
Epoch 2000: PDE Loss = 2.309784, Terminal Loss = 1.576431
Epoch 2500: PDE Loss = 2.346239, Terminal Loss = 1.227793
Epoch 3000: PDE Loss = 2.602135, Terminal Loss = 0.293070
Epoch 3500: PDE Loss = 2.553661, Terminal Loss = 0.386373
Epoch 4000: PDE Loss = 2.540845, Terminal Loss = 1.548357
Epoch 4500: PDE Loss = 2.216487, Terminal Loss = 1.026210
